In [1]:
from dotenv import load_dotenv
import anthropic

load_dotenv()                      
client = anthropic.Anthropic()     #

response = client.messages.create(
    model="claude-haiku-5-5",      
    max_tokens=300,
    messages=[{"role": "user", "content": "In one sentence, what is multicollinearity?"}]
)

print(response.content[0].text)

Multicollinearity is when two or more predictor variables in a regression model are highly correlated with each other, making it difficult to separate their individual effects on the outcome.


In [2]:
import stats_tools as st

disease = st.load_raw()
national_depression = st.get_national_depression(disease)

In [3]:
tools = [
    {
        "name": "depression_by_sex",
        "description": "Returns U.S. national adult depression rates (age-adjusted %) for females and males in a given year, with 95% confidence intervals (LCL, HCL). Data available 2019-2023.",
        "input_schema": {
            "type": "object",
            "properties": {
                "year": {"type": "integer", "description": "Year between 2019 and 2023"}
            },
            "required": ["year"]
        }
    }
]

In [4]:
messages = [{"role": "user", "content": "Is depression higher in women or men in 2022?"}]

response = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=500,
    tools=tools,
    messages=messages
)

print(response.stop_reason)
for block in response.content:
    print(block)

tool_use
ToolUseBlock(id='toolu_01NEL4byLrKYX3dSFhtWf3YH', caller=DirectCaller(type='direct'), input={'year': 2022}, name='depression_by_sex', type='tool_use', toolset_name=None)


In [5]:
tool_call = [b for b in response.content if b.type == "tool_use"][0]

result = st.depression_by_sex(national_depression, **tool_call.input)
result_text = result.to_string(index=False)   # the model reads text, not DataFrames

messages.append({"role": "assistant", "content": response.content})
messages.append({"role": "user", "content": [
    {"type": "tool_result", "tool_use_id": tool_call.id, "content": result_text}
]})

final = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=500,
    tools=tools,
    messages=messages
)
for block in final.content:
    if block.type == "text":
        print(block.text)

In 2022, depression was higher in women than in men. The age-adjusted rate was 26.9% for women (95% CI: 26.5–27.3) and 15.2% for men (95% CI: 14.9–15.5). The confidence intervals don't overlap, so the gap is statistically meaningful.


In [6]:
from agent_tools import TOOLS, SYSTEM_PROMPT, BASELINE_PROMPT, run_tool

def ask(question, max_steps=10, system=SYSTEM_PROMPT):
    messages = [{"role": "user", "content": question}]

    for step in range(max_steps):
        response = client.messages.create(
            model="claude-haiku-5-5",
            max_tokens=2000,
            system=system,
            tools=TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        # No tool requested: the model is done, so print its answer
        if response.stop_reason != "tool_use":
            for block in response.content:
                if block.type == "text":
                    print(block.text)
            return messages

        # Run every tool the model asked for and collect the results
        results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"-> {block.name}({block.input})")
                output = run_tool(block.name, block.input)
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})

        messages.append({"role": "user", "content": results})

    print("Stopped: reached the step limit.")
    return messages

In [7]:
ask("How has the national depression rate changed from 2019 to 2023?");

-> overall_depression({})
The national adult depression rate (age-adjusted %) went down slightly in 2020, then rose each year through 2022, and eased a little in 2023.

| Year | Rate (%) | 95% CI |
|------|----------|--------|
| 2019 | 19.0 | 18.7–19.2 |
| 2020 | 18.6 | 18.3–18.9 |
| 2021 | 19.8 | 19.6–20.1 |
| 2022 | 21.1 | 20.8–21.3 |
| 2023 | 20.7 | 20.4–21.0 |

- **2019 to 2023:** The rate rose by 1.7 points. The intervals don't overlap (18.7–19.2 vs. 20.4–21.0), so this difference is statistically meaningful.
- **2019 to 2021:** The rise to 19.8 is also a clear difference, since the intervals don't overlap.
- **2022 to 2023:** The drop from 21.1 to 20.7 is small, and the intervals overlap (20.8–21.3 vs. 20.4–21.0). The evidence for a real change is inconclusive.
- **2019 to 2020:** The dip to 18.6 is also inconclusive, because the intervals overlap.

These are national trends only. They don't show what caused the change.


In [8]:
ask("Is there a difference in depression between men and women in 2022?");

-> depression_by_sex({'year': 2022})
In 2022, women had a higher depression rate than men.

- **Women:** 26.9% (95% CI: 26.5 to 27.3)
- **Men:** 15.2% (95% CI: 14.9 to 15.5)

The confidence intervals do not overlap, so this is a statistically significant difference. The gap is about 11.7 percentage points. These are age-adjusted national rates, and they describe groups, not individuals. The data cannot tell us why the difference exists.


In [9]:
ask("Which age group has the highest depression rate in 2022?");

-> depression_by_age({'year': 2022})
In 2022, the **18–44** age group had the highest depression rate at **23.7%** (95% CI: 23.3–24.1).

The other groups were:
- **45–64:** 20.0% (95% CI: 19.6–20.4)
- **65+:** 15.5% (95% CI: 15.1–15.9)

The intervals don't overlap, so the gap between the youngest group and the next group is statistically clear. These are crude rates, not age-adjusted, so they don't account for differences in age structure across the groups.


In [10]:
ask("How does depression vary by race/ethnicity in 2022?");

-> depression_by_race({'year': 2022})
In 2022, depression rates (age-adjusted %, 95% CI) varied widely by race/ethnicity:

- **Multiracial, non-Hispanic:** 30.6 (28.7–32.5), the highest
- **White, non-Hispanic:** 24.7 (24.4–25.1)
- **American Indian or Alaska Native, non-Hispanic:** 24.7 (22.6–26.9)
- **Hispanic:** 17.4 (16.7–18.1)
- **Black, non-Hispanic:** 16.9 (16.1–17.6)
- **Native Hawaiian or Other Pacific Islander, non-Hispanic:** 16.4 (13.4–20.0)
- **Asian, non-Hispanic:** 9.9 (8.9–11.0), the lowest

Key comparisons:
- **Multiracial vs. White:** The intervals don't overlap, so the difference is statistically significant.
- **White vs. Black, Hispanic, and Asian:** The intervals don't overlap, so these differences are significant. White rates are higher than Black and Hispanic rates and much higher than Asian rates.
- **White vs. American Indian or Alaska Native:** The intervals overlap, so the evidence is inconclusive.
- **Black vs. Hispanic vs. Native Hawaiian or Pacific Island

In [11]:
ask("Which variable is most strongly correlated with depression across states in 2022?");

-> correlation({'year': 2022})
In 2022, **obesity** has the strongest correlation with state depression rates (r = 0.35, positive). The next strongest is poverty (r = 0.23). Sleeplessness (r = -0.14), checkup (r = 0.08), and inactivity (r = 0.07) are weaker.

These are correlations across 50 states, so they describe associations, not causes. The obesity correlation is moderate at best, and I haven't tested its statistical significance.


In [12]:
ask("Is obesity related to depression across states?");

-> simple_regression({'predictor': 'obesity'})
-> correlation({})
Yes, across states in 2022, higher obesity rates go with higher depression rates. The association is positive and statistically significant, though moderate in strength.

- **Simple regression (50 states, 2022):** Each 1-point increase in a state's obesity rate is associated with about a 0.32-point higher depression rate (95% CI 0.07 to 0.58; p = 0.014).
- **Correlation:** r = 0.35, the strongest positive link of the four correlates with depression except poverty (r = 0.23 for poverty, 0.07 for inactivity, 0.08 for checkup, and -0.14 for sleeplessness).
- **Fit:** R² = 0.119, so obesity accounts for about 12% of the state-to-state variation in depression. Most of the variation is explained by other factors.

This is observational state-level data, so it shows an association, not that obesity causes depression. It also describes states, not individuals. I can run the full five-predictor model if you want to see whether ob

In [13]:
ask("What predicts depression across states in 2022?");

-> full_regression({})
-> correlation({})
-> vif_test({})
-> outlier_check({})
**Short answer:** In 2022, state depression rates were most consistently linked to higher obesity and higher poverty. The full model explains about a quarter of the variation between states.

**Full model (50 states, 2022)**
- Adjusted R² = 0.251 (R² = 0.328), overall F-test p = 0.003

| Predictor | Coefficient | p-value | Notes |
|---|---|---|---|
| Obesity | +0.51 | 0.004 | Strongest, most stable link |
| Poverty | +0.48 | 0.018 | Positive link |
| Inactivity | −0.75 | 0.017 | Sign is surprising (see below) |
| Sleeplessness | −0.34 | 0.060 | Not significant at 0.05 |
| Checkup | +0.27 | 0.085 | Not significant at 0.05 |

**Why the inactivity sign looks odd:** Inactivity alone has almost no correlation with depression (r = 0.07). It is strongly correlated with poverty (r = 0.80) and obesity (r = 0.72). Once those are in the model, its coefficient turns negative. The VIFs are all under 5, with inactivity hi

In [14]:
ask("Is multicollinearity a problem in the regression?");

-> vif_test({})
-> correlation({})
**Not a serious problem, but there is some overlap between two predictors.** All of the 2022 variance inflation factors (VIFs) are below the usual cutoff of 5:

| Predictor | VIF |
|---|---|
| Inactivity | 4.70 |
| Poverty | 3.10 |
| Obesity | 2.13 |
| Sleeplessness | 1.68 |
| Checkup | 1.37 |

Inactivity is close to the cutoff. The strongest overlap is between inactivity and poverty (correlation 0.80), and inactivity and obesity (0.72). Because of this, their individual coefficients are less precise, so I would be cautious about reading much into their separate effects. The predictors are not so redundant that the model's estimates are unreliable.

These results are for 2022 across 50 states. They describe states, not individuals, and any associations are observational rather than causal.


In [15]:
ask("Are the regression results driven by outliers?");

-> outlier_check({})
-> full_regression({})
**No, the results aren't driven by outliers.** Four states have high influence (2022 data), but removing them doesn't change the direction of any coefficient or the main conclusions.

**Influential states** (Cook's distance above the 4/n cutoff of 0.080): Hawaii (0.187), Arkansas (0.140), New Hampshire (0.104), and Nebraska (0.083).

**Full model vs. model without those four states (2022, n = 50 states):**

| Predictor | Coef (full) | p (full) | Coef (without 4) | p (without 4) |
|---|---|---|---|---|
| Inactivity | -0.750 | 0.017 | -0.898 | 0.004 |
| Obesity | 0.513 | 0.004 | 0.537 | 0.001 |
| Sleeplessness | -0.343 | 0.060 | -0.295 | 0.171 |
| Checkup | 0.267 | 0.085 | 0.268 | 0.058 |
| Poverty | 0.483 | 0.018 | 0.532 | 0.006 |

- Inactivity, obesity, and poverty stay significant at p < 0.05 and get slightly stronger without the four states.
- Checkup moves from p = 0.085 to 0.058, so it is borderline in both versions.
- Sleeplessness stays

In [16]:
ask("Are obese people more likely to be depressed?");

-> simple_regression({'predictor': 'obesity'})
-> correlation({})
**Short answer:** This data can't tell us whether obese *people* are more likely to be depressed. It only shows that states with higher obesity rates tend to have higher depression rates.

**What the data shows (2022, 50 states):**
- Correlation between state obesity and state depression: r = 0.35.
- Simple regression: each 1-point increase in a state's obesity rate is associated with about 0.32 more points of depression rate (coefficient 0.32, 95% CI 0.07 to 0.58, p = 0.014).
- R-squared = 0.12, so obesity explains about 12% of the variation in state depression rates. Adjusted R-squared is 0.10.

**Why this doesn't answer the individual question:**
- These are state-level averages. A state-level link doesn't show that obese individuals are more often depressed. This is an ecological comparison, and it can mislead about individuals.
- The data is observational, so it shows association only. It doesn't show that obesity c

In [17]:
ask("Does obesity cause depression?");

-> simple_regression({'predictor': 'obesity'})
-> correlation({})
**No. This data can't show that obesity causes depression.** It shows an association between the two at the state level.

- **Association (2022, 50 states):** States with higher obesity rates tend to have higher depression rates. The correlation is r = 0.35. In a simple regression, each 1-point increase in obesity is linked to about a 0.32-point higher depression rate (coefficient 0.32, p = 0.014). Obesity explains about 12% of the variation in state depression rates (R² = 0.119).
- **Why this isn't causal:** This is observational data. Obesity is closely tied to other state factors, especially inactivity (r = 0.72 with obesity) and poverty (r = 0.62 with obesity). Those factors could account for some or all of the link. The data also can't show which way any effect runs, or whether depression affects weight.
- **Scope:** These results describe states, not individuals. They don't tell us whether a particular person with 

In [18]:
ask("Does being physically inactive protect against depression?");

-> simple_regression({'predictor': 'inactivity'})
-> full_regression({})
-> vif_test({})
-> correlation({})
**No. The data don't show that inactivity protects against depression, and this is observational data, so it can't show cause either way.**

- **Simple comparison:** Across the 50 states in 2022, inactivity alone has essentially no relationship with depression. Coefficient 0.08 (p = 0.62), R² = 0.005, correlation r = 0.07. Higher inactivity is not associated with lower depression here.
- **Multivariable model:** With obesity, poverty, sleeplessness, and checkup included, inactivity has a negative coefficient (−0.75, p = 0.017). Adjusted R² is 0.25. This sign flip is likely an artifact of overlapping predictors. Inactivity correlates strongly with poverty (r = 0.80) and obesity (r = 0.72), and its VIF is 4.7, just under the 5 threshold but high. When those are held constant, the leftover variation in inactivity is small and hard to interpret.
- **Conclusion:** The negative coeffic

In [19]:
ask("Does poverty predict depression in 2023?");

-> simple_regression({'predictor': 'poverty', 'year': 2023})
I can't answer this for 2023. The regression tool returned an error: there is no state-level poverty or sleeplessness data for 2023, so the analysis couldn't run.

I can run the same poverty regression for another year from 2019 to 2022. Would you like that, and if so, which year?


In [20]:
ask("Did depression go down from 2022 to 2023?");

-> overall_depression({})
Probably a small drop, but the data can't confirm it.

- **2022:** 21.1% (95% CI: 20.8 to 21.3)
- **2023:** 20.7% (95% CI: 20.4 to 21.0)

The rate fell by 0.4 percentage points, but the confidence intervals overlap. That means the evidence for a real change is inconclusive. The 2023 rate is still well above 2019 (19.0%).


In [21]:
#testing trap questions without guardrails
ask("Are obese people more likely to be depressed?", system=BASELINE_PROMPT);
ask("Does obesity cause depression?", system=BASELINE_PROMPT);
ask("Does being physically inactive protect against depression?", system=BASELINE_PROMPT);
ask("Does poverty predict depression in 2023?", system=BASELINE_PROMPT);
ask("Did depression go down from 2022 to 2023?", system=BASELINE_PROMPT);

-> simple_regression({'predictor': 'obesity'})
-> correlation({})
**Across U.S. states in 2022, yes, but the data can't tell us whether individual obese people are more likely to be depressed.**

- **Correlation:** States with higher obesity rates tend to have higher depression rates (r = 0.35).
- **Regression:** Each 1-point increase in a state's obesity rate is associated with about a 0.32-point increase in its depression rate (p = 0.014). Obesity explains about 12% of the variation in state depression rates.

This is an ecological, state-level association. It doesn't show that obese individuals are more likely to be depressed. Several things could produce it:

- **Confounding:** Obesity is strongly tied to poverty (r = 0.62), and poverty is also linked to depression (r = 0.23). Other state factors could also play a role.
- **Ecological fallacy:** A pattern across states doesn't necessarily hold for each person within them.
- **Direction:** These cross-sectional data can't show wheth